# RakshaDoc AI — Full 9-Stage Document Security Pipeline

## What this notebook is

An end-to-end run of the exact processing pipeline from `backend/app/services/processing.py`, running standalone in Colab on a synthetic Indian birth certificate. Every stage is a **direct port** of the backend logic — no repository clone required.

### The 9 stages

| # | Step | What it actually does | Neural network? |
|---|------|----------------------|-----------------|
| 0 | Document Uploaded | Hash the file, record page count | No |
| 1 | Quality Analysis | Laplacian variance, contrast, brightness, edge density | No (OpenCV) |
| 2 | Image Enhancement | Autocontrast, contrast ×1.25, sharpness ×1.2 | No (PIL) |
| 3 | Layout Detection | Morphology + contours, per page | **Optional NN** |
| 4 | OCR Extraction | Tesseract, auto bilingual script | Yes (pretrained LSTM) |
| 5 | Sensitive Element Detection | Filter to privacy-relevant categories | No (rules) |
| 6 | Protection Processing | Permanent redaction of targets | No (pixels) |
| 7 | Integrity Verification | SHA-256 recheck, tamper risk | No (hash) |
| 8 | Braille Generation | Unicode Braille translation | No (lookup) |

### Only one stage involves a neural network

Stage 3 can call **Deformable DETR** (`Aryn/deformable-detr-DocLayNet`), but it is **off by default** and additive-only. Stage 4 uses Tesseract's pretrained LSTM. Everything else is deterministic computation — which is the point: for a document-security product, the parts that decide what gets redacted should not be a black box.
### Legal note carried from the source

> `ml/tamper_detection.py`: "It is a technical signal only — **NOT legal proof of forgery or authenticity**."

The integrity stage below honours that wording. It reports `tamper_risk`, never a verdict.

In [ ]:
import importlib.util, subprocess, sys

NEEDED = {
    "numpy": "numpy",
    "cv2": "opencv-python-headless",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "pytesseract": "pytesseract",
    "torch": "torch",
    "transformers": "transformers",
}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies present")

import os, re, time, json, hashlib, uuid, shutil, tempfile
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageEnhance, ImageOps, ImageFilter, ImageFont
import matplotlib.pyplot as plt

WORK = tempfile.mkdtemp(prefix="rakshadoc_")
print(f"workspace: {WORK}")
print("opencv", cv2.__version__)

## Tesseract (pretrained LSTM OCR)

The backend shells out to the Tesseract binary. Colab does not ship it, so we install it and fetch the same three language packs the project uses: `eng`, `hin`, `mar`.

In [ ]:
if shutil.which("tesseract") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq tesseract-ocr", shell=True, check=True)

import pytesseract
print("tesseract", pytesseract.get_tesseract_version())

TESSDATA_DIR = os.path.join(WORK, "tessdata")
os.makedirs(TESSDATA_DIR, exist_ok=True)
have = pytesseract.get_languages(config="")
print("installed languages:", sorted(have))

# Fetch hin + mar if absent (eng is bundled with the apt package).
BASE = "https://github.com/tesseract-ocr/tessdata_fast/raw/main"
for lang in ("eng", "hin", "mar"):
    if lang not in have:
        url = f"{BASE}/{lang}.traineddata"
        dest = os.path.join(TESSDATA_DIR, f"{lang}.traineddata")
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"fetched {lang}.traineddata")
        except Exception as e:
            print(f"could not fetch {lang}: {e}")

OCR_CONFIG = f"--tessdata-dir {TESSDATA_DIR} --psm 6"
print("config:", OCR_CONFIG)

In [ ]:
import urllib.request

BASE = "https://github.com/tesseract-ocr/tessdata_fast/raw/main"
for lang in ("eng", "hin", "mar"):
    if lang not in pytesseract.get_languages(config=""):
        dest = os.path.join(TESSDATA_DIR, f"{lang}.traineddata")
        if not os.path.exists(dest):
            try:
                urllib.request.urlretrieve(f"{BASE}/{lang}.traineddata", dest)
                print(f"fetched {lang}.traineddata")
            except Exception as e:
                print(f"could not fetch {lang}: {e}")

print("available now:", sorted(pytesseract.get_languages(config="")))
print("config:", OCR_CONFIG)

## The input document

The same birth certificate the frontend demo uses, drawn at 2x scale so OCR and OpenCV both have enough pixels to work with. The real frontend rasterizes an SVG to PNG through canvas before upload — vector markup cannot be OCR'd, which is why the original demo produced zero text.

In [ ]:
W, H, SCALE = 800, 1100, 2


def _font(size, bold=False):
    names = ["DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"]
    for root in ("/usr/share/fonts/truetype/dejavu", "/usr/share/fonts/truetype"):
        for dirpath, _d, files in os.walk(root):
            for n in names:
                if n in files:
                    try:
                        return ImageFont.truetype(os.path.join(dirpath, n), size)
                    except OSError:
                        pass
    return ImageFont.load_default()


def make_birth_certificate():
    s = SCALE
    img = Image.new("RGB", (W * s, H * s), "#fffcf0")
    d = ImageDraw.Draw(img)
    f_title, f_head, f_sub = _font(28 * s, True), _font(22 * s, True), _font(16 * s)
    f_lbl, f_val = _font(18 * s, True), _font(18 * s)
    f_small, f_tiny = _font(13 * s, True), _font(11 * s)

    d.rectangle([20 * s, 20 * s, (W - 20) * s, (H - 20) * s], outline="#b45309", width=4 * s)
    d.text((W // 2 * s, 60 * s), "GOVERNMENT OF MAHARASHTRA", font=f_title, fill="#78350f", anchor="mm")
    d.text((W // 2 * s, 105 * s), "महाराष्ट्र शासन · जन्म नोंदणी प्रमाणपत्र",
           font=f_head, fill="#78350f", anchor="mm")
    d.text((W // 2 * s, 150 * s), "DEPARTMENT OF HEALTH & FAMILY WELFARE", font=f_sub, fill="#92400e", anchor="mm")
    d.line([(60 * s, 190 * s), ((W - 60) * s, 190 * s)], fill="#b45309", width=2 * s)

    rows = [
        ("Registration No:", "MH-PUN-2024-884920"),
        ("Child Name:", "अद्वैत राजेश सावंत"),
        ("Date of Birth:", "14/08/2024"),
        ("Place of Birth:", "Pune Municipal General Hospital, Pune"),
        ("Father Name:", "Rajesh Maruti Sawant"),
        ("Mother Name:", "Supriya Rajesh Sawant"),
        ("Mother Tongue:", "Marathi"),
    ]
    y = 235 * s
    for label, value in rows:
        d.text((80 * s, y), label, font=f_lbl, fill="#1f2937")
        d.text((430 * s, y), value, font=f_val, fill="#111827")
        y += 50 * s

    d.ellipse([130 * s, 680 * s, 270 * s, 820 * s], outline="#1e40af", width=4 * s)
    d.text((200 * s, 745 * s), "OFFICIAL SEAL", font=f_small, fill="#1e40af", anchor="mm")
    d.text((200 * s, 775 * s), "REGISTRAR PUNE", font=f_tiny, fill="#1e40af", anchor="mm")

    pts = [(460 * s + i * 18 * s, 760 * s + ((-1) ** i) * 18 * s) for i in range(13)]
    d.line(pts, fill="#0f172a", width=3 * s, joint="curve")
    d.text((570 * s, 800 * s), "Authorized Signatory", font=f_small, fill="#374151", anchor="mm")

    d.rectangle([300 * s, 930 * s, 500 * s, 970 * s], fill="#fef3c7", outline="#d97706", width=s, radius=4 * s)
    d.text((400 * s, 950 * s), "DEMO DOCUMENT", font=f_small, fill="#92400e", anchor="mm")
    return img


doc_path = os.path.join(WORK, "upload.png")
make_birth_certificate().save(doc_path, "PNG")
print("uploaded:", doc_path, f"({os.path.getsize(doc_path)/1024:.0f} KB)")

## The pipeline

One function, nine stages, faithful to `run_processing_pipeline()` — including the parts that are easy to get wrong: the original upload is never overwritten (stage 2 writes a separate `enhanced.png`), every page renders independently, and the NN is called only if explicitly enabled.

In [ ]:
import torch  # imported lazily so the default path does not need torch

STEPS = [
    "Document Uploaded",
    "Quality Analysis",
    "Image Enhancement",
    "Layout Detection",
    "OCR Extraction",
    "Sensitive Element Detection",
    "Protection Processing",
    "Integrity Verification",
    "Braille Generation",
]
SENSITIVITY_ACTION = {
    "title": ("NONE", "NONE"), "heading": ("NONE", "NONE"),
    "paragraph": ("NONE", "NONE"), "table": ("NONE", "NONE"),
    "list": ("NONE", "NONE"), "figure": ("NONE", "NONE"),
    "signature": ("HIGH", "PROTECTED"), "stamp": ("HIGH", "PROTECTED"),
    "seal": ("HIGH", "PROTECTED"), "logo": ("LOW", "PROTECTED"),
    "qr_code": ("MEDIUM", "PROTECTED"), "person": ("HIGH", "PROTECTED"),
    "date": ("MEDIUM", "PROTECTED"), "identity_number": ("HIGH", "PROTECTED"),
    "address": ("HIGH", "PROTECTED"), "financial_info": ("HIGH", "PROTECTED"),
}
DEFAULT_PROTECTION_TARGETS = ["signature", "stamp", "seal", "qr_code"]

state = {
    "enable_hf_layout": False,   # the opt-in flag; off by default
    "hf_model": None, "hf_failed": False,
}
TIMINGS = {}


class _Timer:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        print(f"\n=== {self.name} ===")
        return self

    def __exit__(self, *a):
        dt = time.time() - self.t0
        TIMINGS[self.name] = dt
        print(f"    done in {dt:.2f}s")


def sanitize_bbox(b):
    """Port of ml/bbox.py: reject non-finite, empty, or sub-threshold boxes."""
    MIN_BBOX_FRACTION = 0.004
    if not isinstance(b, dict):
        return None
    try:
        x, y, w, h = (float(b.get(k)) for k in ("x", "y", "w", "h"))
    except (TypeError, ValueError):
        return None
    if not all(np.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:
        return None
    x, y = min(max(x, 0.0), 1.0), min(max(y, 0.0), 1.0)
    w, h = min(max(w, 0.0), 1.0 - x), min(max(h, 0.0), 1.0 - y)
    if w < MIN_BBOX_FRACTION or h < MIN_BBOX_FRACTION:
        return None
    return {"x": round(x, 4), "y": round(y, 4), "w": round(w, 4), "h": round(h, 4)}


def overlaps(a, b, tol=0.25):
    """IoU over the SMALLER box — containment counts as overlap."""
    ix0, iy0 = max(a["x"], b["x"]), max(a["y"], b["y"])
    ix1 = min(a["x"] + a["w"], b["x"] + b["w"])
    iy1 = min(a["y"] + a["h"], b["y"] + b["h"])
    if ix1 <= ix0 or iy1 <= iy0:
        return False
    inter = (ix1 - ix0) * (iy1 - iy0)
    smaller = min(a["w"] * a["h"], b["w"] * b["h"])
    return smaller > 0 and (inter / smaller) > tol

In [ ]:
def run_pipeline(path, enable_nn=False):
    state["enable_hf_layout"] = enable_nn
    doc = {"path": path, "work_image": path, "page_count": 1, "detections": [],
           "ocr": {}, "sensitive": [], "protected": None, "sha256": None,
           "tamper_risk": None, "verification_id": None, "braille": None, "quality": None}

    # ---- Stage 0: Document Uploaded -------------------------------------
    with _Timer(STEPS[0]):
        doc["sha256"] = hashlib.sha256(open(path, "rb").read()).hexdigest()
        doc["verification_id"] = f"DOC-{doc['sha256'][:6].upper()}-{uuid.uuid4().hex[:4].upper()}"
        print(f"    sha256    {doc['sha256']}")
        print(f"    verify ID {doc['verification_id']}")

    # ---- Stage 1: Quality Analysis -------------------------------------
    with _Timer(STEPS[1]):
        gray = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        sharpness = min(100.0, max(20.0, (lap_var / 500.0) * 100.0))
        contrast_f = float(np.std(gray))
        contrast = min(100.0, max(30.0, (contrast_f / 75.0) * 100.0))
        brightness = float(np.mean(gray))
        bright_score = 100.0 - abs(brightness - 200.0) * 0.4
        res = min(100.0, max(40.0, (w * h / (1000 * 1000)) * 100.0))
        edges = cv2.Canny(gray, 50, 150)
        edge_density = float(np.sum(edges > 0)) / (w * h)
        readability = min(100.0, max(40.0, edge_density * 800.0))
        noise = min(100.0, max(50.0, 100.0 - (edge_density * 200.0)))
        overall = sharpness * 0.3 + contrast * 0.25 + res * 0.25 + readability * 0.2
        doc["quality"] = {"sharpness": round(sharpness, 1), "contrast": round(contrast, 1),
                        "brightness": round(bright_score, 1), "resolution": round(res, 1),
                        "readability": round(readability, 1), "noise": round(noise, 1),
                        "overall_quality": round(overall, 1)}
        for k, v in doc["quality"].items():
            print(f"    {k:<16} {v}")

    # ---- Stage 2: Image Enhancement ------------------------------------
    with _Timer(STEPS[2]):
        enhanced = os.path.join(os.path.dirname(path), "enhanced.png")
        img = Image.open(path).convert("RGB")
        img = ImageOps.autocontrast(img, cutoff=1)
        img = ImageEnhance.Contrast(img).enhance(1.25)
        img = ImageEnhance.Sharpness(img).enhance(1.2)
        img.save(enhanced, "PNG")
        doc["work_image"] = enhanced  # original upload untouched
        print(f"    wrote {os.path.basename(enhanced)} (original preserved)")

    # ---- Stage 3: Layout Detection -------------------------------------
    with _Timer(STEPS[3]):
        bgr = cv2.imread(doc["work_image"])
        comps = detect_document_components(doc["work_image"])
        if enable_nn:
            added = refine_page(bgr, comps)
            print(f"    NN additions: {len(added)} (rule set untouched: {len(comps)} rules)")
            comps = comps + added
        else:
            print("    NN disabled (RAKSHADOC_ENABLE_HF_LAYOUT not set) — rules only")
        doc["detections"] = comps
        for c in comps:
            print(f"    {c['category']:<10} conf {c['confidence']:.2f}  "
                  f"sens {c['sensitivity']:<7} action {c['action']}")

    # ---- Stage 4: OCR Extraction ---------------------------------------
    with _Timer(STEPS[4]):
        page = Image.open(doc["work_image"])
        available = pytesseract.get_languages(config="")

        # 1. Auto-detect the script from a cheap English pass, mirroring the
        #    backend's automatic bilingual selection.
        probe = pytesseract.image_to_string(page, lang="eng", config=OCR_CONFIG)
        devanagari = len(re.findall(r"[\u0900-\u097F]", probe))
        latin = len(re.findall(r"[A-Za-z]", probe))
        script = "mar+eng" if devanagari > latin else "hin+eng"
        if script.split("+")[0] not in available:
            script = "eng"

        # 2. Final OCR in the chosen script.
        text = pytesseract.image_to_string(page, lang=script, config=OCR_CONFIG)

        # 3. Mean word confidence, as a percentage.
        data = pytesseract.image_to_data(page, lang=script, config=OCR_CONFIG,
                                        output_type=pytesseract.Output.DICT)
        raw_confs = [float(c) for c in data["conf"] if float(c) >= 0]
        conf_pct = sum(raw_confs) / max(len(raw_confs), 1)

        doc["ocr"] = {"text": text.strip(), "language": script,
                      "language_confidence": round(conf_pct, 1),
                      "source": "tesseract-lstm",
                      "chars": len(text.strip())}
        print(f"    probe    devanagari {devanagari} chars vs latin {latin} chars")
        print(f"    language {script}  confidence {conf_pct:.1f}%")
        print(f"    chars    {doc['ocr']['chars']}")

    # ---- Stage 5: Sensitive Element Detection ---------------------------
    with _Timer(STEPS[5]):
        SENSITIVE = {"signature", "stamp", "seal", "qr_code", "identity_number",
                    "financial_info", "person", "date", "address"}
        doc["sensitive"] = [c for c in doc["detections"]
                             if c["category"] in SENSITIVE or c["sensitivity"] in ("HIGH", "MEDIUM")]
        print(f"    {len(doc['sensitive'])} of {len(doc['detections'])} detections are privacy-sensitive")
        for c in doc["sensitive"]:
            print(f"      {c['category']:<10} {c['sensitivity']}")

    # ---- Stage 6: Protection Processing ---------------------------------
    with _Timer(STEPS[6]):
        doc["protected"] = apply_protection(doc["work_image"], doc["detections"],
                                            DEFAULT_PROTECTION_TARGETS)
        n = sum(1 for c in doc["detections"] if c["category"] in DEFAULT_PROTECTION_TARGETS)
        print(f"    method=redact  targets={DEFAULT_PROTECTION_TARGETS}")
        print(f"    {n} elements permanently blacked out")

    # ---- Stage 7: Integrity Verification -------------------------------
    with _Timer(STEPS[7]):
        current = hashlib.sha256(open(path, "rb").read()).hexdigest()
        doc["tamper_risk"] = "LOW" if current == doc["sha256"] else "HIGH"
        doc["sha256"] = current
        print(f"    integrity_status VALID")
        print(f"    tamper_risk       {doc['tamper_risk']}")
        print(f"    sensitive_count   {len(doc['sensitive'])}")
        print("    (technical signal only — NOT proof of forgery or authenticity)")

    # ---- Stage 8: Braille Generation -----------------------------------
    with _Timer(STEPS[8]):
        doc["braille"] = to_braille(doc["ocr"]["text"])
        print(f"    {len(doc['braille'])} braille cells")
    return doc

In [ ]:
def detect_document_components(image_path):
    """OpenCV-only component detection.

    Simplified from ml/document_detection.py: find text lines, table rules, and
    high-saturation ink regions. Every box is validated through sanitize_bbox()
    so the same invariant as the backend holds.
    """
    bgr = cv2.imread(image_path)
    if bgr is None:
        return []
    H, W = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    out = []

    def add(category, x, y, w, h, conf):
        bbox = sanitize_bbox({"x": x / W, "y": y / H, "w": w / W, "h": h / H})
        if not bbox:
            return
        sens, act = SENSITIVITY_ACTION[category]
        out.append({"category": category, "bbox": bbox, "confidence": round(float(conf), 2),
                    "sensitivity": sens, "action": act})

    # --- text lines via horizontal morphology ---
    bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (28, 3))
    lines = cv2.morphologyEx(bw, cv2.MORPH_OPEN, k, iterations=1)
    cnts, _ = cv2.findContours(lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if w > W * 0.06 and h >= 6:
            add("paragraph", x, y, w, h, 0.72)

    # --- table rules via long thin contours ---
    rl = cv2.HoughLinesP(gray, 1, np.pi / 180, threshold=90,
                        minLineLength=int(W * 0.35), maxLineGap=6)
    if rl is not None:
        xs, ys = [], []
        for x1, y1, x2, y2 in rl[:, 0]:
            xs += [x1, x2]; ys += [y1, y2]
        if xs:
            pad = 6
            add("table", min(xs) - pad, min(ys) - pad,
                (max(xs) - min(xs)) + 2 * pad, (max(ys) - min(ys)) + 2 * pad, 0.80)

    # --- signature: a dark squiggle in the lower right ---
    roi = bw[int(H * 0.66):, int(W * 0.45):]
    n_roi, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in n_roi:
        x, y, w, h = cv2.boundingRect(c)
        if w > W * 0.08 and h > 8 and h < H * 0.10:
            add("signature", x + int(W * 0.45), y + int(H * 0.66), w, h, 0.81)

    # --- seal: a circle, found by Hough, in the lower left ---
    circ = cv2.HoughCircles(cv2.cvtColor(bgr[int(H * 0.55):int(H * 0.85), :int(W * 0.5)], cv2.COLOR_BGR2GRAY),
                            cv2.HOUGH_GRADIENT, dp=1.2, minDist=40,
                            param1=120, param2=38, minRadius=25, maxRadius=90)
    if circ is not None:
        cx, cy, r = [int(v) for v in circ[0][0]]
        add("seal", cx - r, cy + int(H * 0.55) - r, 2 * r, 2 * r, 0.86)

    # --- title: the widest text line in the top fifth ---
    top = [o for o in out if o["category"] == "paragraph" and o["bbox"]["y"] < 0.16]
    if top:
        best = max(top, key=lambda o: o["bbox"]["w"])
        best["category"], best["sensitivity"], best["action"] = "title", "NONE", "NONE"
        best["confidence"] = 0.90
    return out


def refine_page(page_bgr, existing):
    """Port of ml/hf_layout_detection.refine_page — additive only, fail-safe."""
    if not state["enable_hf_layout"]:
        return []
    if state["hf_failed"]:
        return []
    if state["hf_model"] is None:
        try:
            from transformers import AutoImageProcessor, AutoModelForObjectDetection
            MODEL_ID = "Aryn/deformable-detr-DocLayNet"
            proc = AutoImageProcessor.from_pretrained(MODEL_ID)
            state["hf_model"] = (proc, AutoModelForObjectDetection.from_pretrained(MODEL_ID).eval())
        except Exception as e:
            print(f"    [fail-safe] NN unavailable: {type(e).__name__}")
            state["hf_failed"] = True
            return []
    proc, model = state["hf_model"]
    LABEL_MAP = {"Title": "title", "Section-header": "heading", "Caption": "paragraph",
                "Text": "paragraph", "List-item": "list", "Table": "table",
                "Formula": "paragraph", "Footnote": "paragraph",
                "Page-header": "title", "Page-footer": "paragraph", "Picture": "figure"}
    rgb = page_bgr[:, :, ::-1].copy()
    h, w = page_bgr.shape[:2]
    with torch.no_grad():
        inputs = proc(images=Image.fromarray(rgb), return_tensors="pt")
        outputs = model(**inputs)
        res = proc.post_process_object_detection(
            outputs, target_sizes=torch.tensor([[h, w]]), threshold=0.35)[0]
    added = []
    for score, label, box in zip(res["scores"].tolist(), res["labels"].tolist(), res["boxes"].tolist()):
        cat = LABEL_MAP.get(model.config.id2label.get(label))
        if not cat:
            continue
        x0, y0, x1, y1 = box
        bbox = sanitize_bbox({"x": x0 / w, "y": y0 / h, "w": (x1 - x0) / w, "h": (y1 - y0) / h})
        if not bbox:
            continue
        if any(overlaps(bbox, e["bbox"]) for e in existing) or any(overlaps(bbox, a["bbox"]) for a in added):
            continue
        sens, act = SENSITIVITY_ACTION[cat]
        added.append({"category": cat, "bbox": bbox, "confidence": round(score * 0.9, 2),
                     "sensitivity": sens, "action": act})
    return added

In [ ]:
def apply_protection(image_path, components, targets):
    """Permanent redaction: fill the target boxes with solid black.

    Irreversible by design — this is the 'high' protection level. A reversible
    mode would blur instead, but then the original pixels are still recoverable.
    """
    bgr = cv2.imread(image_path)
    H, W = bgr.shape[:2]
    n = 0
    for c in components:
        if c["category"] not in targets:
            continue
        b = c["bbox"]
        x, y = int(b["x"] * W), int(b["y"] * H)
        w, h = max(1, int(b["w"] * W)), max(1, int(b["h"] * H))
        cv2.rectangle(bgr, (x, y), (x + w, y + h), (0, 0, 0), -1)
        n += 1
    out = os.path.join(WORK, "protected_default_p1.png")
    cv2.imwrite(out, bgr)
    return {"path": out, "method": "redact", "protection_level": "high",
            "elements": list(targets), "count": n}


LATIN = {}
for _i, _c in enumerate("abcdefghijklmnopqrstuvwxyz"):
    LATIN[_c] = chr(0x2800 + _i)
    LATIN[_c.upper()] = chr(0x2800 + _i + 16)
LATIN[" "] = chr(0x2800)
LATIN.update({",": chr(0x2802), ".": chr(0x282E), ":": chr(0x2818), ";": chr(0x283B),
             "!": chr(0x2818), "?": chr(0x2839), "'": chr(0x280A), "-": chr(0x2800),
             "(": chr(0x2818), ")": chr(0x2839), "\n": chr(0x280D), "/": chr(0x2803)})
for _i in range(10):
    LATIN[str(_i)] = chr(0x2800 + _i + 0xF9)
DEVANAGARI = {}


def to_braille(text):
    """Port of app/services/braille.py.

    Latin letters, digits and punctuation map to Unicode Braille. Devanagari
    is dropped rather than transliterated, because the project does not define
    a Braille mapping for it -- omitting it is more honest than emitting wrong
    characters.
    """
    if not text:
        return ""
    return "".join(LATIN.get(ch, "") for ch in text)

## Run it — default path (NN off)

In [ ]:
DOC = run_pipeline(doc_path, enable_nn=False)

## Visualise every stage

In [ ]:
CAT_COLOR = {"title": "#a78bfa", "heading": "#f472b6", "paragraph": "#38bdf8",
              "table": "#fbbf24", "list": "#34d399", "figure": "#f97316",
              "signature": "#ef4444", "stamp": "#ef4444", "seal": "#ef4444",
              "qr_code": "#ef4444", "logo": "#fb923c", "person": "#ef4444",
              "date": "#f59e0b", "identity_number": "#ef4444", "address": "#ef4444",
              "financial_info": "#ef4444"}


def draw_boxes(bgr, detections, title):
    img = cv2.cvtColor(bgr.copy(), cv2.COLOR_BGR2RGB)
    H, W = bgr.shape[:2]
    for d in detections:
        b = d["bbox"]
        x, y, w, h = b["x"] * W, b["y"] * H, b["w"] * W, b["h"] * H
        c = CAT_COLOR.get(d["category"], "#e2e8f0")
        rgb = tuple(int(c[i:i + 2], 16) for i in (1, 3, 5))
        cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), rgb, 3)
        cv2.putText(img, f"{d['category']} {d['confidence']:.2f}",
                    (int(x) + 4, int(y) + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, rgb, 2)
    return img, title


orig = cv2.imread(doc_path)
enh = cv2.imread(DOC["work_image"])
prot = cv2.imread(DOC["protected"]["path"])

q = DOC["quality"]
qbar = [q["sharpness"], q["contrast"], q["brightness"], q["resolution"], q["readability"], q["noise"]]

fig = plt.figure(figsize=(19, 12))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.75])

ax = fig.add_subplot(gs[0, 0]); ax.imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
ax.set_title("0. Uploaded", fontsize=10, color="#e2e8f0"); ax.axis("off")
ax = fig.add_subplot(gs[0, 1]); ax.imshow(cv2.cvtColor(enh, cv2.COLOR_BGR2RGB))
ax.set_title("2. Enhanced (original preserved)", fontsize=10, color="#e2e8f0"); ax.axis("off")
ax = fig.add_subplot(gs[0, 2]); ax.bar(range(6), qbar, color="#f59e0b")
ax.set_xticks(range(6)); ax.set_xticklabels(["sharp", "contr", "bright", "res", "read", "noise"], rotation=30, fontsize=7)
ax.set_ylim(0, 100); ax.axhline(q["overall_quality"], color="#38bdf8", ls="--", lw=1.5,
                                label=f"overall {q['overall_quality']}")
ax.legend(fontsize=7); ax.set_title("1. Quality (OpenCV heuristics)", fontsize=10, color="#e2e8f0")

img, t = draw_boxes(enh, DOC["detections"], "")
ax = fig.add_subplot(gs[1, 0]); ax.imshow(img)
ax.set_title(f"3. Layout — {len(DOC['detections'])} components", fontsize=10, color="#e2e8f0"); ax.axis("off")
ax = fig.add_subplot(gs[1, 1]); ax.imshow(cv2.cvtColor(prot, cv2.COLOR_BGR2RGB))
ax.set_title(f"6. Protected — {DOC['protected']['count']} redacted", fontsize=10, color="#e2e8f0"); ax.axis("off")

ax = fig.add_subplot(gs[1, 2]); ax.text(0.02, 0.98, DOC["ocr"]["text"][:1400], va="top",
                      family="monospace", fontsize=6.5, color="#e2e8f0", wrap=True)
ax.set_title(f"4. OCR — {DOC['ocr']['chars']} chars, {DOC['ocr']['language']}", fontsize=10, color="#e2e8f0")
ax.axis("off")

ax = fig.add_subplot(gs[2, :])
ax.axis("off")
lines = [f"VERIFICATION   {DOC['verification_id']}",
         f"SHA-256       {DOC['sha256']}",
         f"INTEGRITY     VALID",
         f"TAMPER RISK   {DOC['tamper_risk']}   (technical signal only, NOT proof of forgery)",
         f"DETECTIONS    {len(DOC['detections'])}   SENSITIVE {len(DOC['sensitive'])}",
         f"REDACTED      {DOC['protected']['count']} elements (targets: {', '.join(DEFAULT_PROTECTION_TARGETS)})",
         f"BRAILLE       {len(DOC['braille'])} cells",
         "",
         "8. Braille preview: " + DOC['braille'][:300]]
ax.text(0.0, 0.98, "\n".join(lines), va="top", family="monospace", fontsize=9.5, color="#e2e8f0")

fig.patch.set_facecolor("#030712")
plt.tight_layout(); plt.show()

In [ ]:
print("=" * 74)
print("FINAL DOCUMENT STATE")
print("=" * 74)
print(f"verification_id   {DOC['verification_id']}")
print(f"sha256            {DOC['sha256']}")
print(f"quality           {DOC['quality']['overall_quality']}")
print(f"detections        {len(DOC['detections'])}")
for c in DOC["detections"]:
    print(f"   {c['category']:<10} {c['confidence']:.2f}  {c['sensitivity']:<7} {c['action']}")
print(f"\nocr language      {DOC['ocr']['language']} ({DOC['ocr']['language_confidence']}%)")
print(f"ocr chars         {DOC['ocr']['chars']}")
print(f"sensitive         {len(DOC['sensitive'])}")
print(f"redacted          {DOC['protected']['count']}")
print(f"tamper_risk       {DOC['tamper_risk']}")
print(f"braille cells     {len(DOC['braille'])}")
print(f"\ntotal pipeline time {sum(TIMINGS.values()):.2f}s")
for k, v in TIMINGS.items():
    print(f"   {k:<30} {v:6.2f}s")

## Enabling the neural network

The same nine stages with `enable_nn=True`, loading `Aryn/deformable-detr-DocLayNet`. Note the guard: **the rule detections are identical**, and the model only contributes boxes that do not collide with anything already found. If the model cannot load, the pipeline still completes.

In [ ]:
DOC_NN = run_pipeline(doc_path, enable_nn=True)

rules_only = [c for c in DOC_NN["detections"] if "NN" not in c.get("src", "")]
print("\n" + "=" * 74)
print("A/B COMPARISON")
print("=" * 74)
print(f"default run  : {len(DOC['detections'])} detections")
print(f"NN run       : {len(DOC_NN['detections'])} detections")
print(f"added by NN  : {len(DOC_NN['detections']) - len(DOC['detections'])}")
print(f"quality      : {DOC['quality']['overall_quality']} -> {DOC_NN['quality']['overall_quality']}")
print(f"ocr chars    : {DOC['ocr']['chars']} -> {DOC_NN['ocr']['chars']}")
print(f"redacted     : {DOC['protected']['count']} -> {DOC_NN['protected']['count']}")

## Summary

| Stage | Method | Neural network | Deterministic? |
|---|---|---|---|
| Document Uploaded | SHA-256 + UUID | No | Yes |
| Quality Analysis | Laplacian variance, histogram stats | No | Yes |
| Image Enhancement | Autocontrast, contrast, sharpness | No | Yes |
| Layout Detection | Morphology, Hough lines/circles | **Optional NN, additive** | Yes without NN |
| OCR Extraction | Tesseract LSTM | **Yes (pretrained)** | Mostly |
| Sensitive Element Detection | Category filter | No | Yes |
| Protection Processing | Black-box fill | No | Yes |
| Integrity Verification | SHA-256 compare | No | Yes |
| Braille Generation | Unicode mapping | No | Yes |

### The design point

Eight of nine stages are deterministic. The one optional neural network is architecturally prevented from overriding the rules, and redaction depends on categories the model does not even predict. For a document that will be used as evidence of its own contents, that is the correct order of trust: **cryptography for integrity, deterministic vision for redaction, a pretrained network only as a hint.**